In [64]:
import pandas as pd

clean_data_path = '../data/clean'

despesas_df = pd.read_parquet(f'{clean_data_path}/despesa-para-exercicio-da-atividade-parlamentar/despesa-para-exercicio-da-atividade-parlamentar.parquet')
partido_df = pd.read_parquet(f'{clean_data_path}/partido/partido.parquet')
projeto_lei_df = pd.read_parquet(f'{clean_data_path}/projeto-lei/projeto_lei.parquet')
deputado_df = pd.read_parquet(f'{clean_data_path}/deputado/deputado.parquet')

In [65]:
print("despesas_df:", despesas_df.columns)
print("partido_df:", partido_df.columns)
print("projeto_lei_df:", projeto_lei_df.columns)
print("deputado_df:", deputado_df.columns)

despesas_df: Index(['nome_parlamentar', 'sigla_partido', 'descricao',
       'descricao_especificacao', 'data_emissao', 'valor_liquido', 'mes',
       'ano', 'deputado_id'],
      dtype='str')
partido_df: Index(['partido_id', 'sigla', 'nome', 'espectro_detalhado', 'espectro_macro'], dtype='str')
projeto_lei_df: Index(['proposicao_id', 'tipo', 'ano', 'autor_deputado_id'], dtype='str')
deputado_df: Index(['nome', 'nome_civil', 'deputado_id', 'sexo'], dtype='str')


In [66]:
import pandas as pd

def preparar_dados(despesas_df, partido_df, projeto_lei_df, deputado_df):
    print("Processando e cruzando os dados...")

    despesas_resumo = (
        despesas_df.groupby("deputado_id")
        .agg(
            valor_total_gasto=("valor_liquido", "sum"),
            sigla_partido=("sigla_partido", "first"),
            nome_parlamentar=("nome_parlamentar", "first"),
        )
        .reset_index()
    )

    projetos_resumo = (
        projeto_lei_df.groupby("autor_deputado_id")
        .agg(total_projetos=("proposicao_id", "count"))
        .reset_index()
    )

    df_completo = deputado_df.merge(
        despesas_resumo, on="deputado_id", how="left"
    ).merge(
        projetos_resumo, left_on="deputado_id", right_on="autor_deputado_id", how="left"
    )

    df_completo = df_completo.dropna(subset=["nome_parlamentar"])
    

    df_completo = df_completo.merge(
        partido_df, left_on="sigla_partido", right_on="sigla", how="left"
    )

    df_completo["espectro_macro"] = df_completo["espectro_macro"].fillna("Indefinido")

    df_completo["total_projetos"] = df_completo["total_projetos"].astype('Int64').fillna(0)
    df_completo["valor_total_gasto"] = df_completo["valor_total_gasto"].fillna(0)

    # Custo por projeto
    df_completo["custo_por_projeto"] = df_completo.apply(
        lambda r: r["valor_total_gasto"] / r["total_projetos"] if r["total_projetos"] > 0 else None,
        axis=1,
    )
    return df_completo

dados_consolidados_df = preparar_dados(despesas_df, partido_df, projeto_lei_df, deputado_df)

Processando e cruzando os dados...


In [67]:
def mostrar_mais_produtivos(df):
    print("\n📈 Top 10 Deputados com Mais Projetos Apresentados:")
    top = df.sort_values(by="total_projetos", ascending=False).head(10)
    colunas = ["nome_parlamentar", "sexo", "sigla_partido", "espectro_macro", "total_projetos", "valor_total_gasto"]
    print(top[colunas].to_string(index=False))


def mostrar_mais_eficientes(df):
    print("\n💰 Top 10 Deputados com Menor Custo por Projeto (Mínimo 1 projeto):")
    filtrado = df[df["total_projetos"] > 0]
    top = filtrado.sort_values(by="custo_por_projeto", ascending=True).head(10)
    colunas = ["nome_parlamentar", "sexo", "sigla_partido", "espectro_macro", "total_projetos", "valor_total_gasto", "custo_por_projeto"]
    print(top[colunas].to_string(index=False))


def mostrar_menos_eficientes(df):
    print("\n💸 Top 10 Deputados com Maior Custo por Projeto (Mínimo 1 projeto):")
    filtrado = df[df["total_projetos"] > 0]
    top = filtrado.sort_values(by="custo_por_projeto", ascending=False).head(10)
    colunas = ["nome_parlamentar", "sexo", "sigla_partido", "espectro_macro", "total_projetos", "valor_total_gasto", "custo_por_projeto"]
    print(top[colunas].to_string(index=False))


def mostrar_resumo_partidos(df):
    print("\n📊 Resumo de Desempenho por Partido e Espectro:")
    resumo = (
        df.groupby(["sigla_partido", "espectro_macro"])
        .agg(
            deputados=("deputado_id", "count"),
            gasto_medio=("valor_total_gasto", "mean"),
            projetos_medios=("total_projetos", "mean"),
        )
        .reset_index()
        .sort_values(by="projetos_medios", ascending=False)
        .head(15)
    )
    print(resumo.to_string(index=False))


def consultar_deputado(df):
    nome_busca = input("\nDigite o nome (ou parte do nome) do parlamentar: ").strip()
    resultado = df[
        df["nome_parlamentar"].str.contains(nome_busca, case=False, na=False)
    ]
    
    if resultado.empty:
        print("\n❌ Nenhum parlamentar encontrado com esse nome.")
    else:
        print(f"\n🔍 Encontrados {len(resultado)} parlamentar(es):")
        colunas = ["nome_parlamentar", "sexo", "sigla_partido", "total_projetos", "valor_total_gasto", "custo_por_projeto"]
        print(resultado[colunas].to_string(index=False))

In [68]:
mostrar_mais_produtivos(dados_consolidados_df)


📈 Top 10 Deputados com Mais Projetos Apresentados:
nome_parlamentar      sexo sigla_partido espectro_macro  total_projetos  valor_total_gasto
     Amom Mandel Masculino  REPUBLICANOS        DIREITA            1205           65340.85
      Duda Ramos Masculino          PODE         CENTRO            1114         1778082.80
  Marcos Tavares Masculino           PDT       ESQUERDA             525         1497149.68
 Jonas Donizette Masculino           PSB       ESQUERDA             473         1759337.90
  Laura Carneiro  Feminino           PSD        DIREITA             271         1699500.34
   José Medeiros Masculino            PL        DIREITA             236         1840306.31
   Kim Kataguiri Masculino        MISSÃO        DIREITA             231          619865.06
      Duarte Jr. Masculino        AVANTE        DIREITA             204         1428401.68
 Capitão Augusto Masculino            PL        DIREITA             202         1881161.28
Pompeo de Mattos Masculino           P

In [69]:
mostrar_mais_eficientes(dados_consolidados_df)


💰 Top 10 Deputados com Menor Custo por Projeto (Mínimo 1 projeto):
nome_parlamentar      sexo sigla_partido espectro_macro  total_projetos  valor_total_gasto  custo_por_projeto
     Amom Mandel Masculino  REPUBLICANOS        DIREITA            1205           65340.85          54.224772
    Ribeiro Neto Masculino SOLIDARIEDADE         CENTRO              93          146443.54        1574.661720
      Duda Ramos Masculino          PODE         CENTRO            1114         1778082.80        1596.124596
   Kim Kataguiri Masculino        MISSÃO        DIREITA             231          619865.06        2683.398528
   Lucas Abrahao Masculino          REDE         CENTRO              30           84345.50        2811.516667
  Marcos Tavares Masculino           PDT       ESQUERDA             525         1497149.68        2851.713676
 Vanderlan Alves Masculino SOLIDARIEDADE         CENTRO              66          215946.18        3271.911818
 Adriana Ventura  Feminino          NOVO        DIRE

In [70]:
mostrar_menos_eficientes(dados_consolidados_df)


💸 Top 10 Deputados com Maior Custo por Projeto (Mínimo 1 projeto):
 nome_parlamentar      sexo sigla_partido espectro_macro  total_projetos  valor_total_gasto  custo_por_projeto
   Keniston Braga Masculino           MDB        DIREITA               1         2024582.94        2024582.940
       Hugo Motta Masculino  REPUBLICANOS        DIREITA               1         1924520.94        1924520.940
   Misael Varella Masculino           PSD        DIREITA               1         1897416.29        1897416.290
Aguinaldo Ribeiro Masculino            PP        DIREITA               1         1751306.32        1751306.320
   Claudio Cajado Masculino            PP        DIREITA               1         1698087.80        1698087.800
Luis Carlos Gomes Masculino  REPUBLICANOS        DIREITA               1         1516300.54        1516300.540
     Josias Gomes Masculino            PT       ESQUERDA               1         1506766.51        1506766.510
    Ricardo Guidi Masculino            PL   

In [71]:
mostrar_resumo_partidos(dados_consolidados_df)


📊 Resumo de Desempenho por Partido e Espectro:
sigla_partido espectro_macro  deputados  gasto_medio  projetos_medios
       MISSÃO        DIREITA          1 6.198651e+05            231.0
         PODE         CENTRO         42 1.015042e+06        56.285714
          PDT       ESQUERDA         25 8.887737e+05            39.68
 REPUBLICANOS        DIREITA         71 9.229105e+05        36.816901
       AVANTE        DIREITA         10 1.027087e+06             35.6
         PSOL       ESQUERDA         17 1.230611e+06        35.235294
SOLIDARIEDADE         CENTRO         14 5.790453e+05        29.071429
           PL        DIREITA        140 1.238062e+06        28.728571
           PV         CENTRO          8 1.088854e+06           25.125
          PSB       ESQUERDA         34 7.525420e+05        24.529412
          PRD         CENTRO          3 5.876748e+05             21.0
           PT       ESQUERDA         90 1.321198e+06        19.811111
          PSD        DIREITA         75 9.

In [72]:
consultar_deputado(dados_consolidados_df)


🔍 Encontrados 1 parlamentar(es):
nome_parlamentar      sexo sigla_partido  total_projetos  valor_total_gasto  custo_por_projeto
   Kim Kataguiri Masculino        MISSÃO             231          619865.06        2683.398528
